In [9]:
reset

Once deleted, variables cannot be recovered. Proceed (y/[n])? y


In [11]:
from netCDF4 import Dataset
import numpy as np
import matplotlib.pyplot as plt
import xarray
import os
import math

In [12]:
# list of ensembles to read in and calc stats on
# ensembles = [ '0051', '0091', '0101', '0111', '0121', 
#              '0131', '0141', '0151', '0161', '0171', 
#              '0181', '0191', '0201', '0211', '0221', 
#              '0231', '0241', '0251', '0261', '0271',
#              '0281', '0291', '0301', '0311', '0321' ]
ensembles = [ '0051', '0091', '0101', '0111' ]

# years = [ str(y) for y in range(2000, 2050) ]
years = [ '2011', '2012' ]

# months = [ '01', '02', '03', '04', '05', '06', '07', 
#           '08', '09', '10', '11', '12' ]
months = [ '05', '11' ]

# classes = [ 'PC1', 'PC2', 'PC3', 'PC4',
#            'PC5', 'PC6', 'PC7', 'IAsuper', 
#            'IA', 'IB', 'IC', 'NIS' ]
classes = [ 'PC5', 'IA' ]

In [13]:
path='/lustre/scratch5/sprice/'
os.chdir(path)

# read in lat and lon information from mesh and find cell indices to restrict data to >=60 North
meshFileName = 'mpaso-IcoswISC30E3r5-restart.nc'
print('reading IN file: ', meshFileName)
dataIn = xarray.open_dataset(meshFileName)

latIn = dataIn.latCell.values
lonIn = dataIn.lonCell.values

rad2deg = 180.0 / np.pi 

# find indices and generate new lat, lon data vectors based on these indices
ind = np.where( (latIn * rad2deg) >= 60.0)[0]

latOut = latIn[ind]
lonOut = lonIn[ind]

# save these for later use (or just re-do this step in other code?)

reading IN file:  mpaso-IcoswISC30E3r5-restart.nc


In [20]:
# set up some strings for manipulating paths and filenames
pathBase='/lustre/scratch5/sprice/'
os.chdir(pathBase)

prefix='v3.LR.historical_'
dirs = '/hist/'
mpassiPrefix = '.mpassi.hist.am.timeSeriesStatsDaily.'
writePath = pathBase+'v3.LR.historical_seaIceRIO_ensembleStats'

In [25]:
print('write processed data path: ', writePath)

write processed data path:  /lustre/scratch5/sprice/v3.LR.historical_seaIceRIO_ensembleStats


In [23]:
pwd

'/lustre/scratch5/.mdt0/sprice'

In [26]:
from polaris_rio import polar_rio

# calc. ensemble averages and stdev by looping over ensembles, for specific months, for specific years
# (resulting outputs will be written to montly netcdf files, as per the single ensemble datasets being
# read in)

for year in years:
    
    for month in months:
     
        for cs in classes:

            first = True
            index = 0

            for ensemble in ensembles:

                # change into relevant ensemble subdir and read in monthly .nc files
                fileBase = pathBase+prefix+ensemble+dirs    
                file = prefix+ensemble+mpassiPrefix+year+'-'+month+'-01.nc'

                # read in file and relevant vars
                print('reading IN file: ', fileBase+file)
                dataIn = xarray.open_dataset(fileBase+file)

                concentration = dataIn.timeDaily_avg_iceAreaCell.values[:,ind]
                thickness = dataIn.timeDaily_avg_iceVolumeCell.values[:,ind]

                ndays = np.size( thickness, axis=0 )
                ncells = np.size( thickness, axis=1 )
                nclasses = len( classes ) 
                nensembles = len( ensembles )

                if first:  

                    # dimension empty storage array 
                    RIOArray = np.zeros([ nensembles, ndays, ncells])
      
                    first = False

                #else:

                # loop over data and fill array w/ RIO values

                # loop over days and cells to assign RIO value for combination of 
                # ice thickness and concentration at that day and location
#                 for day in np.arange(0,ndays):
                for day in np.arange(10,11):

                    for c in np.arange(0,ncells):

                        rio, ice_type, level = polar_rio( vessel_ice_class=cs, 
                                                 ice_thickness_m=thickness[day,c], 
                                                 concentration_tenths=concentration[day,c]*10 
                                                )
                        # append data to pre-dimensioned array
                        RIOArray[index,day,c] = rio
                            
                index = index + 1

            # calc stats (collapse along ensemble dimension) and write to new .nc file
            RIOmedian = np.median( RIOArray, axis=0 )
            RIO5th = np.percentile( RIOArray, 5, axis=0 )
            RIO95th = np.percentile( RIOArray, 95, axis=0 )

            # write ensemble stats back out to (monthly) .nc file
            times = np.arange(1,ndays+1)
            cells = np.arange(1,ncells+1)
            coords = {'Time': times, 'nCells': cells}
            dims = ('Time', 'nCells')

            timeDaily_avg_RIO_ensembleMedian = xarray.DataArray(data=RIOmedian,
                coords=coords, dims=dims, name='timeDaily_avg_RIO_ensembleMedian')
            timeDaily_avg_RIO_ensemble5th = xarray.DataArray(data=RIO5th,
                coords=coords, dims=dims, name='timeDaily_avg_RIO_ensemble5th')
            timeDaily_avg_RIO_ensemble95th = xarray.DataArray(data=RIO95th,
                coords=coords, dims=dims, name='timeDaily_avg_RIO_ensemble95th')

            dataOut = xarray.Dataset({
                'timeDaily_avg_RIO_ensembleMedian':timeDaily_avg_RIO_ensembleMedian,
                'timeDaily_avg_RIO_ensemble5th':timeDaily_avg_RIO_ensemble5th,
                'timeDaily_avg_RIO_ensemble95th':timeDaily_avg_RIO_ensemble95th,
                })

            fileOut = writePath+'/'+prefix+cs+'_EnsStats'+mpassiPrefix+year+'-'+month+'-01.nc'
            print('writing OUT file: ', fileOut)
            dataOut.to_netcdf(fileOut)
                             
            ## clean up before next file is read in
            del dataIn, concentration, thickness, RIOArray, RIOmedian, RIO5th, RIO95th
            del timeDaily_avg_RIO_ensembleMedian, timeDaily_avg_RIO_ensemble5th
            del timeDaily_avg_RIO_ensemble95th, dataOut

            first = True


reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0051/hist/v3.LR.historical_0051.mpassi.hist.am.timeSeriesStatsDaily.2011-05-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0091/hist/v3.LR.historical_0091.mpassi.hist.am.timeSeriesStatsDaily.2011-05-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0101/hist/v3.LR.historical_0101.mpassi.hist.am.timeSeriesStatsDaily.2011-05-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0111/hist/v3.LR.historical_0111.mpassi.hist.am.timeSeriesStatsDaily.2011-05-01.nc
writing OUT file:  /lustre/scratch5/sprice/v3.LR.historical_seaIceRIO_ensembleStats/v3.LR.historical_PC5_EnsStats.mpassi.hist.am.timeSeriesStatsDaily.2011-05-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0051/hist/v3.LR.historical_0051.mpassi.hist.am.timeSeriesStatsDaily.2011-05-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0091/hist/v3.LR.historical_0091.mpassi.hist.am.timeSeriesStatsDaily.2011